# **Football Match Prediction**

This notebook processes football match data to extract *relevant* *features* such as **team form**, **average goals scored**, and **average goals conceded** from the last *five* matches. It then compiles these features into a *DataFrame*, which can be used for training a *predictive* model.

In [ ]:
from google.colab import files


uploaded = files.upload()

Saving E0.csv to E0.csv


In [ ]:
import pandas as pd # Importing data
import io

df = pd.read_csv(io.BytesIO(uploaded['E0.csv']))
print(df)

    Div        Date   Time       HomeTeam     AwayTeam  FTHG  FTAG FTR  HTHG  \
0    E0  15/08/2025  20:00      Liverpool  Bournemouth     4     2   H     1   
1    E0  16/08/2025  12:30    Aston Villa    Newcastle     0     0   D     0   
2    E0  16/08/2025  15:00       Brighton       Fulham     1     1   D     0   
3    E0  16/08/2025  15:00     Sunderland     West Ham     3     0   H     0   
4    E0  16/08/2025  15:00      Tottenham      Burnley     3     0   H     1   
..   ..         ...    ...            ...          ...   ...   ...  ..   ...   
375  E0  24/05/2026  16:00       Man City  Aston Villa     1     2   A     1   
376  E0  24/05/2026  16:00  Nott'm Forest  Bournemouth     1     1   D     1   
377  E0  24/05/2026  16:00     Sunderland      Chelsea     2     1   H     1   
378  E0  24/05/2026  16:00      Tottenham      Everton     1     0   H     1   
379  E0  24/05/2026  16:00       West Ham        Leeds     3     0   H     0   

     HTAG  ... B365CAHH B365CAHA  PCAHH

In [ ]:
df.columns.tolist() # Gives a list of all columns

['Div',
 'Date',
 'Time',
 'HomeTeam',
 'AwayTeam',
 'FTHG',
 'FTAG',
 'FTR',
 'HTHG',
 'HTAG',
 'HTR',
 'Referee',
 'HS',
 'AS',
 'HST',
 'AST',
 'HF',
 'AF',
 'HC',
 'AC',
 'HY',
 'AY',
 'HR',
 'AR',
 'B365H',
 'B365D',
 'B365A',
 'BFDH',
 'BFDD',
 'BFDA',
 'BMGMH',
 'BMGMD',
 'BMGMA',
 'BVH',
 'BVD',
 'BVA',
 'BWH',
 'BWD',
 'BWA',
 'CLH',
 'CLD',
 'CLA',
 'LBH',
 'LBD',
 'LBA',
 'PSH',
 'PSD',
 'PSA',
 'MaxH',
 'MaxD',
 'MaxA',
 'AvgH',
 'AvgD',
 'AvgA',
 'BFEH',
 'BFED',
 'BFEA',
 'B365>2.5',
 'B365<2.5',
 'P>2.5',
 'P<2.5',
 'Max>2.5',
 'Max<2.5',
 'Avg>2.5',
 'Avg<2.5',
 'BFE>2.5',
 'BFE<2.5',
 'AHh',
 'B365AHH',
 'B365AHA',
 'PAHH',
 'PAHA',
 'MaxAHH',
 'MaxAHA',
 'AvgAHH',
 'AvgAHA',
 'BFEAHH',
 'BFEAHA',
 'B365CH',
 'B365CD',
 'B365CA',
 'BFDCH',
 'BFDCD',
 'BFDCA',
 'BMGMCH',
 'BMGMCD',
 'BMGMCA',
 'BVCH',
 'BVCD',
 'BVCA',
 'BWCH',
 'BWCD',
 'BWCA',
 'CLCH',
 'CLCD',
 'CLCA',
 'LBCH',
 'LBCD',
 'LBCA',
 'PSCH',
 'PSCD',
 'PSCA',
 'MaxCH',
 'MaxCD',
 'MaxCA',
 'AvgCH',
 'Avg

We decided to do **feature engineering** because if you try to use the current data to make a decision tree, the data that happens after kickoff (such as *corners* and *shots*) is actually **leaked** to the model *prior to prediction*. That is why it is a lot smarter to train the decision tree on data where it only has access to things before kick-off (such as form, goals scored and conceded in the last five games, etc).

In [ ]:
df = df[[
    "Date",
    "HomeTeam",
    "AwayTeam",
    "FTHG",
    "FTAG",
    "FTR"
]].copy()

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], dayfirst = True)
df = (
    df.sort_values('Date').reset_index(drop = True) #The drop = true resets the index of each row.
)

In [ ]:
feature_rows = []
team_history = {}
season_points = {}

In [ ]:
def get_last5_stats(history):
  if (len(history) == 0):
    return [0, 0, 0, 0, 0, 0, 0, 0] # Return 8 zeros for all expected stats

  last5 = history[-5:] # Get last 5 matches
  form = sum(match['Points'] for match in last5)

  avg_scored = sum(match['goals_scored'] for match in last5) / len(last5)
  avg_conceded = sum(match['goals_conceded'] for match in last5) / len(last5)
  avg_goal_diff = sum(match['goals_scored'] - match['goals_conceded'] for match in last5) / len(last5)

  wins = sum(1 for match in last5 if match['Points'] == 3)
  win_pct = wins / len(last5)

  draws = sum(1 for match in last5 if match['Points'] == 1)
  draw_pct = draws / len(last5)

  clean_sheets = sum(1 for match in last5 if match['goals_conceded'] == 0)
  clean_sheet_pct = clean_sheets / len(last5)

  failed_to_score = sum(1 for match in last5 if match['goals_scored'] == 0)
  failed_to_score_pct = failed_to_score / len(last5)

  return [form, avg_scored, avg_conceded, avg_goal_diff, win_pct, draw_pct, clean_sheet_pct, failed_to_score_pct]

In [ ]:
def get_position(team, season_points):
  sorted_teams = sorted(season_points.items(), key = lambda x : x[1], reverse = True) # x[1] means that you are sorting according to points and in decending order.

  for position, (team_name, points) in enumerate(sorted_teams, start = 1): # Position is just i, and start = 1 just makes the league position.
    if team_name == team:
      return position

  return len(sorted_teams) + 1 # Case for start of season where there will no positions.

In [ ]:
def get_venue_form(history, venue):
  venue_matches = [match for match in history if match["venue"] == venue]
  last5 = venue_matches[-5:]

  if len(last5) == 0:
    return 0

  return sum(match["Points"] for match in last5)


In [ ]:
for _, row in df.iterrows():  # returns (index, row), _ means ignore the index.
  home_team = row['HomeTeam']
  away_team = row['AwayTeam']

  home_season_points = season_points.get(home_team, 0)  # dictionary.get(key, default_value)
  away_season_points = season_points.get(away_team, 0)

  home_position = get_position(home_team, season_points)
  away_position = get_position(away_team, season_points)

  home_history = team_history.get(home_team, []) # Either get a empty [] or the team already exists.
  away_history = team_history.get(away_team, []) # Give me the value for this key. If it doesn't exist, return the default value.

  home_home_form = get_venue_form(home_history, "H")
  away_away_form = get_venue_form(away_history, "A")

  home_form, home_avg_scored, home_avg_conceded, home_goal_diff, home_win_pct, home_draw_pct, home_clean_sheet_pct, home_failed_to_score_pct = get_last5_stats(home_history)
  away_form, away_avg_scored, away_avg_conceded, away_goal_diff, away_win_pct, away_draw_pct, away_clean_sheet_pct, away_failed_to_score_pct = get_last5_stats(away_history)

  feature_rows.append({
      "Home_team" : home_team,
      "Away_team" : away_team,
      "Home_form" : home_form,
      "Away_form" : away_form,
      "Home_avg_scored" : home_avg_scored,
      "Away_avg_scored" : away_avg_scored,
      "Home_avg_conceded" : home_avg_conceded,
      "Away_avg_conceded" : away_avg_conceded,
      "Home_goal_diff" : home_goal_diff,
      "Away_goal_diff" : away_goal_diff,
      "Goal_diff_difference" : home_goal_diff - away_goal_diff,
      "Home_win_pct" : home_win_pct,
      "Away_win_pct" : away_win_pct,
      "Home_draw_pct" : home_draw_pct,
      "Away_draw_pct" : away_draw_pct,
      "Home_clean_sheet_pct" : home_clean_sheet_pct,
      "Away_clean_sheet_pct" : away_clean_sheet_pct,
      "Home_failed_to_score_pct" : home_failed_to_score_pct,
      "Away_failed_to_score_pct" : away_failed_to_score_pct,
      "Home_season_points" : home_season_points,
      "Away_season_points" : away_season_points,
      "Points Differenece" : home_season_points - away_season_points,
      "home_position" : home_position,
      "away_position" : away_position,
      "Position_difference" : away_position - home_position,
      "Home_home_form" : home_home_form,
      "Away_away_form" : away_away_form,
      "Venue_form_difference" : home_home_form - away_away_form,
      "Position_gap" : abs(home_position - away_position),
      "Points_ratio" : (home_season_points + 1) / (away_season_points + 1),
      "Result" : row['FTR']
  })

  home_goals = row['FTHG'] # Returns goals scored by home team.
  away_goals = row['FTAG'] # Returns goals scored by away team.

  if home_goals > away_goals:
    home_points = 3
    away_points = 0

  elif away_goals > home_goals:
    home_points = 0
    away_points = 3

  else:
    home_points = 1
    away_points = 1

  season_points[home_team] = season_points.get(home_team, 0) + home_points
  season_points[away_team] = season_points.get(away_team, 0) + away_points

  if home_team not in team_history:
    team_history[home_team] = [] # Initialize list for home team if not present

  if away_team not in team_history:
    team_history[away_team] = [] # Initialize list for away team if not present

  team_history[home_team].append({
      "Points" : home_points,
      "goals_scored" : home_goals,
      "goals_conceded" : away_goals,
      "venue" : "H"
  })

  team_history[away_team].append({
      "Points" : away_points,
      "goals_scored" : away_goals,
      "goals_conceded" : home_goals,
      "venue" : "A"
  })

In [ ]:
feature_df = pd.DataFrame(feature_rows) # Make the final feature

In [ ]:
feature_df.head()

,Home_team,Away_team,Home_form,Away_form,Home_avg_scored,Away_avg_scored,Home_avg_conceded,Away_avg_conceded,Home_goal_diff,Away_goal_diff,...,Points Differenece,home_position,away_position,Position_difference,Home_home_form,Away_away_form,Venue_form_difference,Position_gap,Points_ratio,Result
0,Liverpool,Bournemouth,0,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,1,1,0,0,0,0,0,1.0,H
1,Aston Villa,Newcastle,0,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,3,3,0,0,0,0,0,1.0,D
2,Brighton,Fulham,0,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,5,5,0,0,0,0,0,1.0,D
3,Sunderland,West Ham,0,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,7,7,0,0,0,0,0,1.0,H
4,Tottenham,Burnley,0,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,9,9,0,0,0,0,0,1.0,H


In [ ]:
feature_df.to_csv("Football_features_25-26 (V2).csv", index=False)

In [ ]:
from google.colab import files

files.download("Football_features_25-26 (V2).csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>